# TB Portals — LOCAL 01 · Build manifest

Builds  from the TB Portals Published Imaging zip.
Run this notebook once; the manifest is reused by notebooks 03 and 04.

In [1]:
# ── Paths — edit these to match your machine ──────────────────────────────
import os, sys
from pathlib import Path

# Repo root: auto-detected from this notebook's location (notebooks/ is one level down)
REPO_DIR = str(Path(os.path.abspath("")).parent)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Local working directory (manifest + checkpoints go here, inside the repo)
WORK = str(Path(REPO_DIR) / "local_work")
os.makedirs(f"{WORK}/data/processed", exist_ok=True)

# TB Portals annotation zip (CSVs live inside this zip)
ZIP_PATH = r"E:\Archive\TB_Portals_Published_Imaging_Data_August_2023.zip"

# PRIMARY: point at the extracted directory after running:
#   powershell -ExecutionPolicy Bypass -File scripts\extract_cxr_dcm.ps1
IMAGE_ROOTS = [
    r"E:\dcm",
]

# ALTERNATIVE (if extraction not done yet) — index DICOMs directly from zips:
# IMAGE_ROOTS = [
#     r"E:\Archive\TB_Portals_Published_CXRs_August_2023_Amended.zip",
#     r"E:\March2025\TB_Portals_CXRs_March_2025",
#     r"E:\March2026\TB_Portals_CXRs_March_2026",
# ]

MANIFEST = f"{WORK}/data/processed/tbportals_manifest.csv"
print("REPO_DIR:", REPO_DIR)
print("WORK:    ", WORK)
print("IMAGE_ROOTS:", IMAGE_ROOTS)


REPO_DIR: c:\Users\mabdu_8h1ndel\OneDrive - Higher Education Commission\Desktop\Uni\sem_6\DL\proj\dl-project-codebase
WORK:     c:\Users\mabdu_8h1ndel\OneDrive - Higher Education Commission\Desktop\Uni\sem_6\DL\proj\dl-project-codebase\local_work
IMAGE_ROOTS: ['E:\\dcm']


## Step 1 — Peek inside the zip
Lists all CSV files so you can confirm the table names and choose the right ones.

In [2]:
from src.data.tbportals import list_csvs_in_zip
import zipfile, pandas as pd

csvs = list_csvs_in_zip(ZIP_PATH)
print("CSVs in zip:")
for c in csvs:
    print(" ", c)


CSVs in zip:
  TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CTs_August_2023.csv
  TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CT_Annotations_August_2023.csv
  TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CT_UIIP_Annotations_August_2023.csv
  TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CXRs_August_2023.csv
  TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CXR_Manual_Annotations_August_2023.csv
  TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CXR_Qure_Annotations_August_2023.csv
  TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CXR_Zhiying_Annotations_August_2023.csv


## Step 2 — Check column names
Print the columns of both key tables so you can verify or override them.

In [3]:
from src.data.tbportals import extract_csv_from_zip

# Extract both CSVs to inspect columns
out_csv_dir = f"{WORK}/data/processed/csv_inspect"
cxr_csv  = extract_csv_from_zip(ZIP_PATH, out_csv_dir, csv_name="CXRs")
ann_csv  = extract_csv_from_zip(ZIP_PATH, out_csv_dir, csv_name="CXR_Manual_Annotations")

cxr_df = pd.read_csv(cxr_csv, nrows=3)
ann_df = pd.read_csv(ann_csv, nrows=3)

print("=== CXR metadata columns ===")
print(list(cxr_df.columns))
print("=== Annotations columns ===")
print(list(ann_df.columns))
print("If the ALP or cavity column names differ from defaults, pass")
print("column_map={'alp': 'actual_alp_col', 'cavity': 'actual_cavity_col'} in Step 4.")

[tbportals] extracted -> c:\Users\mabdu_8h1ndel\OneDrive - Higher Education Commission\Desktop\Uni\sem_6\DL\proj\dl-project-codebase\local_work\data\processed\csv_inspect\TB_Portals_CXRs_August_2023.csv
[tbportals] extracted -> c:\Users\mabdu_8h1ndel\OneDrive - Higher Education Commission\Desktop\Uni\sem_6\DL\proj\dl-project-codebase\local_work\data\processed\csv_inspect\TB_Portals_CXR_Manual_Annotations_August_2023.csv
=== CXR metadata columns ===
['patient_id', 'condition_id', 'age_of_onset', 'sex', 'type_of_resistance', 'comorbidity', 'country', 'outcome', 'imagingstudy_id', 'imaging_date', 'series_instance_content_url', 'series_modality_cd', 'cxr_outlier']
=== Annotations columns ===
['condition_id', 'imagingstudy_id', 'sextant', 'collapse', 'smallcavities', 'mediumcavities', 'largecavities', 'isanylargecavitybelongtoamultisextantcavity', 'canmultiplecavitiesbeseen', 'lowgroundglassdensity', 'mediumdensity', 'highdensity', 'smallnodules', 'mediumnodules', 'largenodules', 'hugenodul

## Step 2b — Diagnose image paths (run if Step 3 gives 0 DICOMs)
Lists first-level contents of each root and sample paths so you can set IMAGE_ROOTS correctly.

In [4]:
import os, zipfile

# Sanity-check IMAGE_ROOTS before running Step 3
print("=== IMAGE_ROOTS contents ===")
for root in IMAGE_ROOTS:
    print(f"\n{root}")
    if not os.path.exists(root):
        print("  NOT FOUND — run scripts/extract_cxr_dcm.ps1 first")
        continue
    if root.lower().endswith(".zip"):
        with zipfile.ZipFile(root) as z:
            dcm_entries = [e for e in z.namelist() if e.endswith(".dcm")]
        print(f"  zip: {len(dcm_entries)} DCM entries")
        if dcm_entries: print(f"  sample: {dcm_entries[0]}")
    else:
        dcm_count = 0; zip_count = 0; sample_dcm = None
        for dp, _, fns in os.walk(root):
            for fn in fns:
                if fn.lower().endswith(".dcm"):
                    dcm_count += 1
                    if sample_dcm is None:
                        sample_dcm = os.path.join(dp, fn)
                elif fn.lower().endswith(".zip"):
                    zip_count += 1
        if dcm_count:
            print(f"  {dcm_count} loose DCMs")
            print(f"  sample: {sample_dcm}")
        elif zip_count:
            print(f"  0 loose DCMs but {zip_count} zips (will be indexed in-zip)")
        else:
            print("  EMPTY — no DCMs or zips found")


=== IMAGE_ROOTS contents ===

E:\dcm
  18900 loose DCMs
  sample: E:\dcm\f272d01b-9947-47c2-8e3e-2624942a6472\1.2.826.0.1.3680043.2.1125.1.84606778210724879644609873186626875\1.2.826.0.1.3680043.2.1125.1.95390918585275787567807231276516690\2.25.218600615034370593969253480973286238009.dcm


## Step 3 — Build DICOM index
Walks all image roots and indexes every  file. This takes a few minutes on first run but only needs to be done once per session.

**Skip this cell if you already have  in memory.**

In [5]:
from src.data.tbportals import build_dicom_index

dicom_index = build_dicom_index(IMAGE_ROOTS)
print(f"Index ready: {len(dicom_index)//2} unique DICOM files")


[tbportals] indexed 18900 DICOMs under E:\dcm
[tbportals] DICOM index total: 18900 unique DICOMs
Index ready: 18900 unique DICOM files


## Step 4 — Build manifest
Joins the two CSVs on  and resolves each DICOM path.

If Step 2 showed different column names for ALP or cavity, uncomment and fill in .

In [6]:
from src.data.tbportals import build_manifest_from_tbportals_zip

# Uncomment and fill in if your column names differ from defaults:
# COLUMN_MAP_OVERRIDE = {
#     "alp":    "actual_alp_column_name",
#     "cavity": "actual_cavity_column_name",
# }
COLUMN_MAP_OVERRIDE = None

df = build_manifest_from_tbportals_zip(
    ZIP_PATH,
    IMAGE_ROOTS,
    MANIFEST,
    column_map=COLUMN_MAP_OVERRIDE,
    dicom_index=dicom_index,
)


[tbportals] CXR metadata CSV:  TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CXRs_August_2023.csv
[tbportals] Annotations CSV:   TB_Portals_Published_Imaging_Data_August_2023/TB_Portals_CXR_Manual_Annotations_August_2023.csv
[tbportals] CXR metadata cols:  ['patient_id', 'condition_id', 'age_of_onset', 'sex', 'type_of_resistance', 'comorbidity', 'country', 'outcome', 'imagingstudy_id', 'imaging_date', 'series_instance_content_url', 'series_modality_cd', 'cxr_outlier']
[tbportals] Annotations cols:   ['condition_id', 'imagingstudy_id', 'sextant', 'collapse', 'smallcavities', 'mediumcavities', 'largecavities', 'isanylargecavitybelongtoamultisextantcavity', 'canmultiplecavitiesbeseen', 'lowgroundglassdensity', 'mediumdensity', 'highdensity', 'smallnodules', 'mediumnodules', 'largenodules', 'hugenodules', 'isanycalcifiedorpartiallycalcifiednoduleexist', 'isanynoncalcifiednoduleexist', 'isanyclusterednoduleexists', 'aremultiplenoduleexists', 'lowgroundglassdensityactivefreshnodul

## Step 5 — Verify

In [7]:
from src.data.tbportals import load_manifest

df = load_manifest(MANIFEST)
print(df.dtypes)
print(df.head(3))

for c in ["Romania", "Moldova", "Kazakhstan"]:
    assert c in set(df["country"]), f"Held-out country missing: {c}"

print("Manifest OK ->", MANIFEST)
print(f"Total images: {len(df)},  patients: {df.patient_id.nunique()}")


image_id          str
image_path        str
patient_id        str
country           str
alp_0_100     float64
cavity          int64
dtype: object
                                            image_id  \
0  f272d01b-9947-47c2-8e3e-2624942a6472/1.2.826.0...   
1  7d7efbbb-6823-4951-ad72-df151cc4688e/1.2.826.0...   
2  f8ba953e-e0de-439c-98a1-e78e2b3693c2/1.2.826.0...   

                                          image_path  \
0  E:\dcm\f272d01b-9947-47c2-8e3e-2624942a6472\1....   
1  E:\dcm\7d7efbbb-6823-4951-ad72-df151cc4688e\1....   
2  E:\dcm\f8ba953e-e0de-439c-98a1-e78e2b3693c2\1....   

                             patient_id  country  alp_0_100  cavity  
0  3c3ce6bc-c2f5-4d8d-9c0d-17a3e16ff1c5  Georgia       10.0       0  
1  ba1525e0-9853-455a-8e10-537c43fad5c3  Ukraine       40.0       1  
2  fe905324-7ead-4b78-8bea-3bd83f758de3  Georgia       30.0       1  
Manifest OK -> c:\Users\mabdu_8h1ndel\OneDrive - Higher Education Commission\Desktop\Uni\sem_6\DL\proj\dl-project-codebase\l